In [ ]:
!pip install torch transformers accelerate anthropic

In [2]:
import os
from getpass import getpass

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")
if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

Hugging Face token:  ········
Anthropic API key:  ········


In [3]:
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
import anthropic
client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"], max_retries=10)

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B", add_prefix_space=True)
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B",
    dtype=torch.float16,
    device_map="auto"
)
device = next(model.parameters()).device
device

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

device(type='cuda', index=0)

In [9]:
import json
import re
import pandas as pd

CORPUS_PATH = "../results/generated_corpus.jsonl"
GREEDY_CORPUS_PATH = "../results/generated_corpus_greedy.jsonl"
TWO_JUDGES_CORPUS_PATH = "../results/generated_corpus_two_judges.jsonl"


def load_corpus(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def save_corpus(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

In [7]:
def clean_token(token_str: str) -> str:
    return token_str.strip().lstrip("Ġ▁ ")


@torch.inference_mode()
def generate_greedy_cover(model, tokenizer, prompt, min_tokens=24):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    past_key_values = None
    generated_ids = []
    while True:
        outputs = model(input_ids=input_ids, past_key_values=past_key_values, use_cache=True)
        past_key_values = outputs.past_key_values
        picked_id = outputs.logits[0, -1, :].argmax().item()
        if picked_id == tokenizer.eos_token_id:
            break
        generated_ids.append(picked_id)
        input_ids = torch.tensor([[picked_id]], device=device)
        if len(generated_ids) > min_tokens and re.fullmatch(r"[.?!]+", clean_token(tokenizer.decode(picked_id))):
            break
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

In [8]:
greedy_corpus = []
stego = [rec for rec in load_corpus(CORPUS_PATH) if rec["label"] == "stego"]
for idx, rec in enumerate(stego):
    cover = {key: None for key in rec}
    cover.update(prompt=rec["prompt"], text=generate_greedy_cover(model, tokenizer, rec["prompt"]), label="cover")
    greedy_corpus += [rec, cover]
    if (idx + 1) % 25 == 0:
        print(f"  ... {idx + 1}/{len(stego)} prompts processed")
save_corpus(greedy_corpus, GREEDY_CORPUS_PATH)

print(pd.DataFrame(greedy_corpus)["label"].value_counts())

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  ... 25/272 prompts processed
  ... 50/272 prompts processed
  ... 75/272 prompts processed
  ... 100/272 prompts processed
  ... 125/272 prompts processed
  ... 150/272 prompts processed
  ... 175/272 prompts processed
  ... 200/272 prompts processed
  ... 225/272 prompts processed
  ... 250/272 prompts processed
label
stego    272
cover    272
Name: count, dtype: int64


In [10]:
def chat_with_claude(text: str) -> str:
    prompt = (
        f"Rate the following text on a scale from 1 to 100, where 100 means perfect in terms of mistakes, coherence, fluency, grammar, naturalness, and truthfulness, and 1 means it's completely incoherent, full of mistakes or unnatural. As a response just return the number! Text to rate:\n\n\"{text}\"")
    response = client.messages.create(
        model="claude-opus-4-7",
        max_tokens=16,
        system="You are a helpful and precise text quality evaluator.",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return next(block.text for block in response.content if block.type == "text").strip()

In [11]:
corpus = load_corpus(TWO_JUDGES_CORPUS_PATH if os.path.exists(TWO_JUDGES_CORPUS_PATH) else CORPUS_PATH)
for rec in corpus:
    rec.setdefault("score_claude", None)
stego = [rec for rec in corpus if rec["label"] == "stego"]
for idx, rec in enumerate(stego):
    if rec["score_claude"] is None:
        rec["score_claude"] = chat_with_claude(rec["text"])
        save_corpus(corpus, TWO_JUDGES_CORPUS_PATH)
    if (idx + 1) % 50 == 0:
        print(f"  ... {idx + 1}/{len(stego)} stego texts rated")

claude_score = pd.to_numeric(pd.Series([rec["score_claude"] for rec in stego]), errors="coerce")
print(f"Claude Opus 4.7 score of stego texts: mean = {claude_score.mean():.2f}, median = {claude_score.median():g}")
print(f"Texts without a numeric score: {claude_score.isna().sum()}")

  ... 50/272 stego texts rated
  ... 100/272 stego texts rated
  ... 150/272 stego texts rated
  ... 200/272 stego texts rated
  ... 250/272 stego texts rated
Claude Opus 4.7 score of stego texts: mean = 60.36, median = 70
Texts without a numeric score: 0
